# test_grid_core.ipynb：逐行演算 `envs/grid/grid_core.py`

这个 notebook 用一份很小的 fake pandapower 网络，解释 `GridCore` 从输入功率到潮流结果字段的计算路径。

每个代码 cell 都会输出一张解释表：

1. 给出具体输入。
2. 展示对应代码或公式。
3. 打印中间输出。
4. 用中文解释这一步的意义。

最后一个 cell 会跑一次真实 simbench/GridCore，只展示实际运行的 shape 和少量数值，不用于手算。

## 0. 准备环境和公共工具

Notebook 不一定从项目根目录启动，所以先用 `find_project_root(...)` 找到仓库根目录，并加入 `sys.path`，避免出现 `No module named 'envs'`。

In [1]:
from pathlib import Path
import sys
from types import SimpleNamespace

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "envs").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError(f"Cannot find project root from {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from envs.grid.grid_core import GridCore, GridStepResult, _AgentBusBinding

rows = [
    [1, "PROJECT_ROOT = find_project_root(Path.cwd())", str(Path.cwd()), str(PROJECT_ROOT), "找到包含 envs/configs 的仓库根目录。"],
    [2, "sys.path.insert(0, str(PROJECT_ROOT))", "项目根目录", sys.path[0], "让 notebook 可以直接 import envs.grid.grid_core。"],
    [3, "from envs.grid.grid_core import ...", "grid_core.py", "导入成功", "后续 cell 会调用真实 GridCore 方法。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

,步骤,代码,本例输入,本例输出,解释
0,1,PROJECT_ROOT = find_project_root(Path.cwd()),D:\GithubProject\MADRL_ESS\tests\testnotebook,D:\GithubProject\MADRL_ESS,找到包含 envs/configs 的仓库根目录。
1,2,"sys.path.insert(0, str(PROJECT_ROOT))",项目根目录,D:\GithubProject\MADRL_ESS,让 notebook 可以直接 import envs.grid.grid_core。
2,3,from envs.grid.grid_core import ...,grid_core.py,导入成功,后续 cell 会调用真实 GridCore 方法。


## 1. `GridStepResult`：一次潮流计算的输出容器

`GridStepResult` 不做计算，它只是把一次潮流运行后的物理量放在同一个对象里。

In [2]:
result = GridStepResult(
    converged=True,
    vm_pu=np.array([0.94, 1.07, 1.00], dtype=np.float32),
    line_loading_pct=np.array([80.0, 125.0], dtype=np.float32),
    trafo_loading_pct=np.array([130.0], dtype=np.float32),
    v_violation=np.array([0.01, 0.02], dtype=np.float32),
    trafo_p_signed_kw=np.array([-12.0], dtype=np.float32),
    psi_v_raw=0.0005,
    psi_line_raw=0.0625,
    psi_trafo_raw=0.09,
)

rows = [
    [1, "converged=True", "潮流成功", result.converged, "告诉环境这一步 pandapower 是否收敛。"],
    [2, "vm_pu.shape", "3 个 bus 电压", result.vm_pu.shape, "每个 bus 一个电压标幺值。"],
    [3, "line_loading_pct.shape", "2 条 line", result.line_loading_pct.shape, "每条线路一个负载百分比。"],
    [4, "trafo_loading_pct.shape", "1 台 trafo", result.trafo_loading_pct.shape, "每台变压器一个负载百分比。"],
    [5, "v_violation", "agent bus 电压 [0.94, 1.07]", result.v_violation.tolist(), "低于 0.95 是 0.01，高于 1.05 是 0.02。"],
    [6, "psi_*", "全网违规", (result.psi_v_raw, result.psi_line_raw, result.psi_trafo_raw), "reward 使用这些全局安全惩罚。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

,步骤,代码,本例输入,本例输出,解释
0,1,converged=True,潮流成功,True,告诉环境这一步 pandapower 是否收敛。
1,2,vm_pu.shape,3 个 bus 电压,"(3,)",每个 bus 一个电压标幺值。
2,3,line_loading_pct.shape,2 条 line,"(2,)",每条线路一个负载百分比。
3,4,trafo_loading_pct.shape,1 台 trafo,"(1,)",每台变压器一个负载百分比。
4,5,v_violation,"agent bus 电压 [0.94, 1.07]","[0.009999999776482582, 0.019999999552965164]",低于 0.95 是 0.01，高于 1.05 是 0.02。
5,6,psi_*,全网违规,"(0.0005, 0.0625, 0.09)",reward 使用这些全局安全惩罚。


## 5. `_write_agent_injections(...)`：把 kW 注入覆盖到 pandapower 表

这个方法每次直接覆盖 agent bus 的 `load/sgen`：

- `p_inject_kw > 0`：写入 `sgen.p_mw`，`load.p_mw` 写 0。
- `p_inject_kw < 0`：写入 `load.p_mw`，`sgen.p_mw` 写 0。
- `q_mvar` 统一写成 0。

因此它同时完成“清掉旧注入”和“写入当前注入”这两件事。

In [3]:
def make_fake_core() -> GridCore:
    net = SimpleNamespace(
        bus=pd.DataFrame(index=[101, 102, 999]),
        line=pd.DataFrame(index=[0, 1]),
        trafo=pd.DataFrame({"sn_mva": [0.4]}, index=[0]),
        load=pd.DataFrame(
            {"bus": [101, 102], "p_mw": [0.010, 0.020], "q_mvar": [0.0, 0.0]},
            index=[10, 11],
        ),
        sgen=pd.DataFrame(
            {"bus": [101, 102], "p_mw": [0.001, 0.002], "q_mvar": [0.0, 0.0]},
            index=[20, 21],
        ),
        res_bus=pd.DataFrame({"vm_pu": [0.94, 1.07, 1.00]}, index=[101, 102, 999]),
        res_line=pd.DataFrame({"loading_percent": [80.0, 125.0]}, index=[0, 1]),
        res_trafo=pd.DataFrame({"loading_percent": [130.0], "p_hv_mw": [-0.012]}, index=[0]),
    )

    core = object.__new__(GridCore)
    core.deployments = []
    core.grid_cfg = SimpleNamespace(v_min_pu=0.95, v_max_pu=1.05, line_max_loading_pct=100.0, pf_solver="nr")
    core.n_agents = 2
    core.agent_bus_ids = [101, 102]
    core.net = net
    core.n_buses = 3
    core.n_lines = 2
    core.n_trafos = 1
    core._agent_bus_pos = np.array([0, 1], dtype=np.int64)
    core._bus_rows = [
        _AgentBusBinding(load_idx=10, sgen_idx=20),
        _AgentBusBinding(load_idx=11, sgen_idx=21),
    ]
    core._last_valid = GridCore._make_zero_result(core)
    core.last_pf_error = ""
    return core


core = make_fake_core()
rows = [
    [1, "core.agent_bus_ids", "两个 agent", core.agent_bus_ids, "agent 0 接到 bus 101，agent 1 接到 bus 102。"],
    [2, "core._agent_bus_pos", "bus index [101, 102, 999]", core._agent_bus_pos.tolist(), "结果表里第 0、1 行对应两个 agent bus。"],
    [3, "core._bus_rows[0]", "bus 101", core._bus_rows[0], "记录 bus 101 对应的 load/sgen 行。"],
    [4, "core._bus_rows[1]", "bus 102", core._bus_rows[1], "记录 bus 102 对应的 load/sgen 行。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

,步骤,代码,本例输入,本例输出,解释
0,1,core.agent_bus_ids,两个 agent,"[101, 102]",agent 0 接到 bus 101，agent 1 接到 bus 102。
1,2,core._agent_bus_pos,"bus index [101, 102, 999]","[0, 1]",结果表里第 0、1 行对应两个 agent bus。
2,3,core._bus_rows[0],bus 101,"_AgentBusBinding(load_idx=10, sgen_idx=20)",记录 bus 101 对应的 load/sgen 行。
3,4,core._bus_rows[1],bus 102,"_AgentBusBinding(load_idx=11, sgen_idx=21)",记录 bus 102 对应的 load/sgen 行。


## 3. `GridCore.step(...)` 的输入 shape 合同

`GridCore.step` 现在要求 `p_batt_kw` 和 `base_load_kw` 都是一维向量，并且长度等于 `n_agents`。

In [4]:
core = make_fake_core()
base_load_kw = np.array([5.0, 3.0], dtype=np.float32)
bad_p_batt_kw = np.array([2.0], dtype=np.float32)

try:
    GridCore.step(core, p_batt_kw=bad_p_batt_kw, base_load_kw=base_load_kw)
except ValueError as exc:
    error_message = str(exc)

rows = [
    [1, "expected_shape = (self.n_agents,)", "n_agents = 2", (2,), "期望每个 agent 一个功率值。"],
    [2, "p_batt_kw.shape", bad_p_batt_kw.tolist(), bad_p_batt_kw.shape, "这里只有 1 个值，和 2 个 agent 不匹配。"],
    [3, "raise ValueError(...) if shape mismatch", "shape 不一致", error_message, "在写入电网前失败，避免静默少写或错写。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

,步骤,代码,本例输入,本例输出,解释
0,1,"expected_shape = (self.n_agents,)",n_agents = 2,"(2,)",期望每个 agent 一个功率值。
1,2,p_batt_kw.shape,[2.0],"(1,)",这里只有 1 个值，和 2 个 agent 不匹配。
2,3,raise ValueError(...) if shape mismatch,shape 不一致,"GridCore.step expected p_batt_kw shape (2,), r...",在写入电网前失败，避免静默少写或错写。


## 4. 从负荷和电池动作得到电网注入 `p_inject_kw`

`GridCore.step` 先计算：

```python
p_inject_kw = -(base_load_kw + p_batt_kw)
```

这里的符号约定是：正的 `net_load` 表示从电网取电，所以写入 pandapower 时变成负注入。

In [5]:
base_load_kw = np.array([5.0, 3.0], dtype=np.float32)
p_batt_kw = np.array([2.0, -1.0], dtype=np.float32)
net_load_kw = base_load_kw + p_batt_kw
p_inject_kw = -net_load_kw

rows = [
    [1, "base_load_kw", "agent 0/1", base_load_kw.tolist(), "基础净负荷，单位 kW。"],
    [2, "p_batt_kw", "agent 0 充电 2，agent 1 放电 1", p_batt_kw.tolist(), "正值增加取电，负值减少取电。"],
    [3, "net_load_kw = base_load_kw + p_batt_kw", "[5, 3] + [2, -1]", net_load_kw.tolist(), "agent 0 最终取电 7 kW，agent 1 最终取电 2 kW。"],
    [4, "p_inject_kw = -net_load_kw", "-[7, 2]", p_inject_kw.tolist(), "负注入会被写成 pandapower 的 load。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码/公式", "本例输入", "本例输出", "解释"]))

,步骤,代码/公式,本例输入,本例输出,解释
0,1,base_load_kw,agent 0/1,"[5.0, 3.0]",基础净负荷，单位 kW。
1,2,p_batt_kw,agent 0 充电 2，agent 1 放电 1,"[2.0, -1.0]",正值增加取电，负值减少取电。
2,3,net_load_kw = base_load_kw + p_batt_kw,"[5, 3] + [2, -1]","[7.0, 2.0]",agent 0 最终取电 7 kW，agent 1 最终取电 2 kW。
3,4,p_inject_kw = -net_load_kw,"-[7, 2]","[-7.0, -2.0]",负注入会被写成 pandapower 的 load。


## 5. `_write_agent_injections(...)`：把 kW 注入覆盖到 pandapower 表

这个方法每次直接覆盖 agent bus 的 `load/sgen`：

- `p_inject_kw > 0`：写入 `sgen.p_mw`，`load.p_mw` 写 0。
- `p_inject_kw < 0`：写入 `load.p_mw`，`sgen.p_mw` 写 0。
- `q_mvar` 统一写成 0。

因此它同时完成“清掉旧注入”和“写入当前注入”这两件事。

In [6]:
core = make_fake_core()
p_inject_kw = np.array([-7.0, 4.0], dtype=np.float32)

before_load = core.net.load[["p_mw", "q_mvar"]].copy()
before_sgen = core.net.sgen[["p_mw", "q_mvar"]].copy()
GridCore._write_agent_injections(core, p_inject_kw)
after_load = core.net.load[["p_mw", "q_mvar"]].copy()
after_sgen = core.net.sgen[["p_mw", "q_mvar"]].copy()

rows = [
    [1, "p_mw = float(p_kw) / 1e3", "-7 kW", -7.0 / 1000.0, "kW 转成 MW。"],
    [2, "load.p_mw = max(0, -p_mw)", "p_mw = -0.007", float(after_load.at[10, "p_mw"]), "负注入表示负荷，所以 bus 101 的 load 写成 0.007 MW。"],
    [3, "sgen.p_mw = max(0, p_mw)", "p_mw = -0.007", float(after_sgen.at[20, "p_mw"]), "负注入不是发电，所以 sgen 写成 0。"],
    [4, "p_mw = 4 / 1000", "4 kW", 0.004, "正注入表示发电或反送。"],
    [5, "sgen.p_mw = max(0, p_mw)", "p_mw = 0.004", float(after_sgen.at[21, "p_mw"]), "bus 102 的 sgen 写成 0.004 MW。"],
    [6, "load.p_mw = max(0, -p_mw)", "p_mw = 0.004", float(after_load.at[11, "p_mw"]), "正注入不是负荷，所以 load 写成 0。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码/公式", "本例输入", "本例输出", "解释"]))

,步骤,代码/公式,本例输入,本例输出,解释
0,1,p_mw = float(p_kw) / 1e3,-7 kW,-0.007,kW 转成 MW。
1,2,"load.p_mw = max(0, -p_mw)",p_mw = -0.007,0.007,负注入表示负荷，所以 bus 101 的 load 写成 0.007 MW。
2,3,"sgen.p_mw = max(0, p_mw)",p_mw = -0.007,0.000,负注入不是发电，所以 sgen 写成 0。
3,4,p_mw = 4 / 1000,4 kW,0.004,正注入表示发电或反送。
4,5,"sgen.p_mw = max(0, p_mw)",p_mw = 0.004,0.004,bus 102 的 sgen 写成 0.004 MW。
5,6,"load.p_mw = max(0, -p_mw)",p_mw = 0.004,0.000,正注入不是负荷，所以 load 写成 0。


## 6. `reset(...)`：用零注入清空 agent bus

旧版本先恢复再写入，现在合并成 `_write_agent_injections(...)`：

- `reset(...)` 写入全零注入。
- `step(...)` 写入当前步注入。

因为建网时已经把原始静态功率清零，所以零注入就是 agent bus 的干净状态。

In [7]:
core = make_fake_core()
GridCore._write_agent_injections(core, np.array([-7.0, 4.0], dtype=np.float32))
changed_load = core.net.load["p_mw"].tolist()
changed_sgen = core.net.sgen["p_mw"].tolist()
GridCore.reset(
    core,
    base_load_kw=np.array([5.0, 3.0], dtype=np.float32),
    base_pv_kw=np.array([1.0, 2.0], dtype=np.float32),
)
reset_load = core.net.load["p_mw"].tolist()
reset_sgen = core.net.sgen["p_mw"].tolist()

rows = [
    [1, "_write_agent_injections([-7, 4])", "当前步注入", (changed_load, changed_sgen), "load/sgen 被改成当前步的状态。"],
    [2, "reset(base_load_kw, base_pv_kw)", "reset 的两个输入当前不参与写网", "调用成功", "reset 只负责清空 agent 注入和错误状态。"],
    [3, "_write_agent_injections(np.zeros(n_agents))", "[0, 0] kW", reset_load, "零注入后 load.p_mw 全部为 0。"],
    [4, "sgen.p_mw = max(0, 0)", "[0, 0] kW", reset_sgen, "零注入后 sgen.p_mw 全部为 0。"],
    [5, "last_pf_error", "reset 后", repr(core.last_pf_error), "reset 会清空上一次潮流错误。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

,步骤,代码,本例输入,本例输出,解释
0,1,"_write_agent_injections([-7, 4])",当前步注入,"([0.007, 0.0], [0.0, 0.004])",load/sgen 被改成当前步的状态。
1,2,"reset(base_load_kw, base_pv_kw)",reset 的两个输入当前不参与写网,调用成功,reset 只负责清空 agent 注入和错误状态。
2,3,_write_agent_injections(np.zeros(n_agents)),"[0, 0] kW","[0.0, 0.0]",零注入后 load.p_mw 全部为 0。
3,4,"sgen.p_mw = max(0, 0)","[0, 0] kW","[0.0, 0.0]",零注入后 sgen.p_mw 全部为 0。
4,5,last_pf_error,reset 后,'',reset 会清空上一次潮流错误。


## 7. `_extract_result(...)`：从 pandapower 结果表得到违规指标

这个方法读取 `res_bus`、`res_line`、`res_trafo`，计算 agent 电压违规和全网 `psi_*` 惩罚。

In [8]:
core = make_fake_core()
result = GridCore._extract_result(core, converged=True)

expected_agent_v_violation = [0.95 - 0.94, 1.07 - 1.05]
expected_psi_v = (0.95 - 0.94) ** 2 + (1.07 - 1.05) ** 2
expected_psi_line = ((125.0 - 100.0) / 100.0) ** 2
expected_psi_trafo = ((130.0 - 100.0) / 100.0) ** 2

rows = [
    [1, "vm_pu = res_bus['vm_pu']", "[0.94, 1.07, 1.00]", result.vm_pu.tolist(), "读取所有 bus 电压。"],
    [2, "agent_vm_pu = take(vm_pu, [0, 1])", "agent bus 位置 [0, 1]", [0.94, 1.07], "只取 agent 对应 bus 的电压算逐 agent 违规。"],
    [3, "v_violation", "[0.94, 1.07], 限制 [0.95, 1.05]", result.v_violation.tolist(), "分别得到约 0.01 和 0.02。"],
    [4, "psi_v_raw = sum(bus_v_excess ** 2)", "0.01^2 + 0.02^2", result.psi_v_raw, f"手算结果 {expected_psi_v:.6f}。"],
    [5, "psi_line_raw", "line 125% 超过 100%", result.psi_line_raw, f"((125-100)/100)^2 = {expected_psi_line:.4f}。"],
    [6, "psi_trafo_raw", "trafo 130% 超过 100%", result.psi_trafo_raw, f"((130-100)/100)^2 = {expected_psi_trafo:.4f}。"],
    [7, "trafo_p_signed_kw", "p_hv_mw = -0.012", result.trafo_p_signed_kw.tolist(), "MW 乘 1000，得到 -12 kW。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码/公式", "本例输入", "本例输出", "解释"]))

,步骤,代码/公式,本例输入,本例输出,解释
0,1,vm_pu = res_bus['vm_pu'],"[0.94, 1.07, 1.00]","[0.9399999976158142, 1.0700000524520874, 1.0]",读取所有 bus 电压。
1,2,"agent_vm_pu = take(vm_pu, [0, 1])","agent bus 位置 [0, 1]","[0.94, 1.07]",只取 agent 对应 bus 的电压算逐 agent 违规。
2,3,v_violation,"[0.94, 1.07], 限制 [0.95, 1.05]","[0.009999990463256836, 0.020000100135803223]",分别得到约 0.01 和 0.02。
3,4,psi_v_raw = sum(bus_v_excess ** 2),0.01^2 + 0.02^2,0.0005,手算结果 0.000500。
4,5,psi_line_raw,line 125% 超过 100%,0.0625,((125-100)/100)^2 = 0.0625。
5,6,psi_trafo_raw,trafo 130% 超过 100%,0.09,((130-100)/100)^2 = 0.0900。
6,7,trafo_p_signed_kw,p_hv_mw = -0.012,[-12.0],MW 乘 1000，得到 -12 kW。


## 8. `step(...)` 成功路径：写入、运行潮流、提取结果

下面让 `_runpp()` 什么都不做，相当于假设 pandapower 已经成功收敛。这样可以专注看 `step` 的控制流。

In [9]:
core = make_fake_core()
core._runpp = lambda: None
base_load_kw = np.array([5.0, 3.0], dtype=np.float32)
p_batt_kw = np.array([2.0, -1.0], dtype=np.float32)
result = GridCore.step(core, p_batt_kw=p_batt_kw, base_load_kw=base_load_kw)

rows = [
    [1, "shape check", "p_batt/base_load 都是 (2,)", "通过", "输入长度和 n_agents 一致。"],
    [2, "p_inject_kw = -(base_load + p_batt)", "-[5+2, 3-1]", [-(5 + 2), -(3 - 1)], "得到 [-7, -2] kW。"],
    [3, "_write_agent_injections(...)之后 load.p_mw", "[-7, -2] kW", core.net.load["p_mw"].tolist(), "两个 agent 都是取电，所以写入 load。"],
    [4, "_runpp()", "这里用 lambda 模拟成功", "没有异常", "真实运行时这里会调用 pandapower.runpp。"],
    [5, "_extract_result(converged=True)", "fake res_* 表", result.converged, "成功路径返回 converged=True 的结果。"],
    [6, "last_pf_error", "成功路径", repr(core.last_pf_error), "成功后错误字符串清空。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

,步骤,代码,本例输入,本例输出,解释
0,1,shape check,"p_batt/base_load 都是 (2,)",通过,输入长度和 n_agents 一致。
1,2,p_inject_kw = -(base_load + p_batt),"-[5+2, 3-1]","[-7, -2]","得到 [-7, -2] kW。"
2,3,_write_agent_injections(...)之后 load.p_mw,"[-7, -2] kW","[0.007, 0.002]",两个 agent 都是取电，所以写入 load。
3,4,_runpp(),这里用 lambda 模拟成功,没有异常,真实运行时这里会调用 pandapower.runpp。
4,5,_extract_result(converged=True),fake res_* 表,True,成功路径返回 converged=True 的结果。
5,6,last_pf_error,成功路径,'',成功后错误字符串清空。


## 9. `step(...)` 失败路径：返回上一轮有效结果

如果 pandapower 抛异常，`GridCore` 不直接中断 episode，而是返回 `_last_valid` 的拷贝，并把 `converged` 改成 `False`。

In [10]:
core = make_fake_core()
core._last_valid = GridStepResult(
    converged=True,
    vm_pu=np.array([1.01, 1.02, 1.03], dtype=np.float32),
    line_loading_pct=np.array([10.0, 20.0], dtype=np.float32),
    trafo_loading_pct=np.array([30.0], dtype=np.float32),
    v_violation=np.array([0.0, 0.0], dtype=np.float32),
    trafo_p_signed_kw=np.array([5.0], dtype=np.float32),
)


def fail_runpp() -> None:
    raise RuntimeError("demo pf failed")


core._runpp = fail_runpp
failed = GridCore.step(
    core,
    p_batt_kw=np.zeros(2, dtype=np.float32),
    base_load_kw=np.zeros(2, dtype=np.float32),
)

rows = [
    [1, "_runpp()", "模拟 RuntimeError", core.last_pf_error, "错误信息记录到 last_pf_error。"],
    [2, "dataclasses.replace(_last_valid, converged=False)", "上一轮有效 vm [1.01,1.02,1.03]", failed.converged, "返回结果标记为未收敛。"],
    [3, "failed.vm_pu", "上一轮有效 vm", failed.vm_pu.tolist(), "物理字段沿用上一轮有效值。"],
    [4, "failed.trafo_p_signed_kw", "上一轮有效 trafo 功率", failed.trafo_p_signed_kw.tolist(), "安全投影和 info 仍能拿到 shape 稳定的数据。"],
]
display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

,步骤,代码,本例输入,本例输出,解释
0,1,_runpp(),模拟 RuntimeError,RuntimeError: demo pf failed,错误信息记录到 last_pf_error。
1,2,"dataclasses.replace(_last_valid, converged=False)","上一轮有效 vm [1.01,1.02,1.03]",False,返回结果标记为未收敛。
2,3,failed.vm_pu,上一轮有效 vm,"[1.0099999904632568, 1.0199999809265137, 1.029...",物理字段沿用上一轮有效值。
3,4,failed.trafo_p_signed_kw,上一轮有效 trafo 功率,[5.0],安全投影和 info 仍能拿到 shape 稳定的数据。


## 10. 真实 simbench/GridCore 最小运行

最后跑一次真实 `GridCore`。这一步不用于心算，只确认真实 simbench 网络下的输入输出 shape 和收敛状态。

In [11]:
try:
    from configs.experiment_config import GridConfig
    from envs.grid.deployments import AgentDeployment
    from envs.grid.net_builder import build_simbench_net

    sb_code = "1-LV-rural1--0-sw"
    net = build_simbench_net(sb_code)
    chosen_buses = [int(bus_id) for bus_id in list(net.load["bus"].unique())[:2]]
    deployments = [
        AgentDeployment(bus_id=bus_id, battery_capacity_kwh=5.0, battery_power_kw=2.5)
        for bus_id in chosen_buses
    ]
    grid_cfg = GridConfig(
        sb_code=sb_code,
        pf_solver="nr",
        v_min_pu=0.95,
        v_max_pu=1.05,
        line_max_loading_pct=100.0,
    )
    real_core = GridCore(deployments, grid_cfg)
    real_result = real_core.step(
        p_batt_kw=np.zeros(2, dtype=np.float32),
        base_load_kw=np.array([0.5, 0.25], dtype=np.float32),
    )
    rows = [
        [1, "chosen_buses", "simbench load buses 前 2 个", chosen_buses, "真实 agent bus。"],
        [2, "real_result.converged", "真实 pandapower.runpp", real_result.converged, "真实潮流是否收敛。"],
        [3, "real_result.vm_pu.shape", "真实 bus 数", real_result.vm_pu.shape, "输出每个 bus 的电压。"],
        [4, "real_result.line_loading_pct.shape", "真实 line 数", real_result.line_loading_pct.shape, "输出每条线路的 loading。"],
        [5, "real_result.v_violation", "两个 agent", real_result.v_violation.tolist(), "每个 agent 一个电压违规值。"],
        [6, "real_core.last_pf_error", "真实运行", repr(real_core.last_pf_error), "成功时为空；失败时会记录异常类型和信息。"],
    ]
except Exception as exc:
    rows = [
        [1, "真实 simbench/GridCore 运行", "当前环境", type(exc).__name__, "真实依赖不可用或潮流失败时，这里展示错误而不中断 notebook。"],
        [2, "错误信息", "exception", str(exc), "pytest 才是正式回归验证入口。"],
    ]

display(pd.DataFrame(rows, columns=["步骤", "代码", "本例输入", "本例输出", "解释"]))

D:\SOFTWARE\miniconda\envs\MADRL_ESS\Lib\site-packages\simbench\converter\csv_pp_converter.py:874: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_data[output_name] = pd.concat([output_data[output_name], input_data[


D:\SOFTWARE\miniconda\envs\MADRL_ESS\Lib\site-packages\simbench\converter\csv_pp_converter.py:874: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_data[output_name] = pd.concat([output_data[output_name], input_data[


D:\SOFTWARE\miniconda\envs\MADRL_ESS\Lib\site-packages\simbench\converter\csv_pp_converter.py:874: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_data[output_name] = pd.concat([output_data[output_name], input_data[


D:\SOFTWARE\miniconda\envs\MADRL_ESS\Lib\site-packages\simbench\converter\csv_pp_converter.py:874: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_data[output_name] = pd.concat([output_data[output_name], input_data[


D:\SOFTWARE\miniconda\envs\MADRL_ESS\Lib\site-packages\simbench\converter\csv_pp_converter.py:874: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  output_data[output_name] = pd.concat([output_data[output_name], input_data[


,步骤,代码,本例输入,本例输出,解释
0,1,chosen_buses,simbench load buses 前 2 个,"[9, 7]",真实 agent bus。
1,2,real_result.converged,真实 pandapower.runpp,True,真实潮流是否收敛。
2,3,real_result.vm_pu.shape,真实 bus 数,"(15,)",输出每个 bus 的电压。
3,4,real_result.line_loading_pct.shape,真实 line 数,"(13,)",输出每条线路的 loading。
4,5,real_result.v_violation,两个 agent,"[0.0, 0.0]",每个 agent 一个电压违规值。
5,6,real_core.last_pf_error,真实运行,'',成功时为空；失败时会记录异常类型和信息。
